<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# What to learn next

**[What to learn next](https://learning.nextia-ai.com/courses/recommendations/m10/what-to-learn-next/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 10, case study 4 (optional) · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare popularity, co-occurrence, a Markov sequence method and a prerequisite rule on synthetic progress data from a learning platform, measure next-lesson recall and how often a suggestion breaks a prerequisite, and decide what an "Up next" block should show.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in less than a minute after the packages install |
| **Needs** | An internet connection for the setup cells. No account. Not the course project. |
| **You should know** | Recall@k and segments, from [Evaluate by segment](https://learning.nextia-ai.com/courses/recommendations/m03/evaluate-by-segment/). Markov suggestions, from [Sessions, sequences and LLMs, briefly](https://learning.nextia-ai.com/courses/recommendations/m07/sessions-sequences-and-llms-briefly/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab and Kaggle were not run. |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/what-to-learn-next/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L04-what-to-learn-next/what-to-learn-next-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: NumPy 2.5.4 and pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It makes a folder `learn-next` with a `data` folder in it. It downloads two files from the course's labs repository (about 2 MB) and checks each file's checksum (SHA-256). It needs an internet connection and no account, and takes a few seconds.

You should see `Ready:` and the two files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `learn-next` (in Colab: the **Files** panel on the left), then run the cell again.

**The data is synthetic.** Our code made it for this case study, with a model of how learners behave (`make_data.py` in the same labs folder). "Ridgeway Learning" is an invented learning platform. No real learner is in this data. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L04-what-to-learn-next/dataset.md) explains how it was made. Licence: CC0 1.0.

In [ ]:
# Setup: download the data into learn-next/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/M10-L04-what-to-learn-next/"
FILES = {
    "lessons.csv": "b3b5b2c32cbdcd9f58d4d566a2a80ca5a90d3190cccff7098ad9ed7c23cd76fc",
    "completions.csv.gz": "a554af287d2b7d7f46be262f1272daf9d6e08e3b9374d29126d27d9d06f293e0",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("learn-next").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder learn-next and run again.")

os.chdir(WORK)
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.2f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the case study's tools. Read the comments if you like; the lesson page explains each one.

- `read()` loads the catalogue and the completions. Lessons get small numbers (`item`) so that they fit in an array.
- `Catalogue` holds each lesson's course, position and prerequisites. `eligible(done, cat)` says which lessons a learner **can take**: not done yet, with every prerequisite done.
- `Split(events, cat, start)` is the time split: everything known before `start`, and the learners who complete a lesson in the two weeks from `start`, with the lesson they really completed next.
- `top_k` and `fill` turn scores into lists of 3 lessons, never one the learner has done.
- `evaluate(split, lists)` measures the lists: recall@3 of the next lesson with a 95% interval, NDCG@3, the **break rate** (the share of suggestions whose prerequisites the learner has not done), coverage, novelty, diversity, and recall by segment.
- `expect(...)` and `expect_hash(...)` check your answers without showing them.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, lists and measures (the same code as the lesson page).
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

K = 3                                                     # the "Up next" block shows 3 lessons


PERIODS = {"validation": "2026-01-05", "test": "2026-01-19", "new input": "2026-02-02"}


DAYS = 14                                                 # each period is two weeks


SEGMENTS = ["new", "in a course", "between courses"]


def read(folder="data"):
    """The catalogue (one row per lesson, in course order) and the completions, with lesson numbers (`item`)."""
    lessons = pd.read_csv(f"{folder}/lessons.csv", keep_default_na=False)
    events = pd.read_csv(f"{folder}/completions.csv.gz", parse_dates=["completed_at"])
    events["item"] = events["lesson"].map({l: i for i, l in enumerate(lessons["lesson"])})
    return lessons, events


class Catalogue:
    """The lessons as arrays: course, position, and the prerequisites as lesson numbers."""

    def __init__(self, lessons):
        self.ids = lessons["lesson"].to_numpy()
        self.n = len(lessons)
        number = {l: i for i, l in enumerate(self.ids)}
        self.course = lessons["course"].to_numpy()
        self.position = lessons["position"].to_numpy()
        self.optional = lessons["optional"].astype(str).eq("True").to_numpy()
        self.pre = [[number[p] for p in s.split(";") if p] for s in lessons["prerequisites"]]
        self.courses = list(dict.fromkeys(self.course))
        self.first = {c: int(np.flatnonzero(self.course == c)[0]) for c in self.courses}
        self.last = {c: int(np.flatnonzero(self.course == c)[-1]) for c in self.courses}


def eligible(done, cat):
    """True where the learner has not done the lesson and has done all its prerequisites."""
    ok = ~done
    for j, pre in enumerate(cat.pre):
        if pre:
            ok[:, j] &= done[:, pre].all(axis=1)
    return ok


class Split:
    """The time split: everything known before `start` (`history`), and the learners who complete a lesson in the
    two weeks from `start`. For each of them: the lessons done before (`done`), the last lesson done (`last`, -1 for
    none), the lesson they really complete next (`next`, the first completion in the period) and their segment."""

    def __init__(self, events, cat, start, days=DAYS):
        self.cat = cat
        self.start = pd.Timestamp(start)
        end = self.start + pd.Timedelta(days=days)
        self.history = events[events["completed_at"] < self.start]
        period = events[(events["completed_at"] >= self.start) & (events["completed_at"] < end)]
        first = period.sort_values("completed_at", kind="stable").drop_duplicates("learner")
        self.learners = first["learner"].to_numpy()
        self.next = first["item"].to_numpy()
        row = {l: r for r, l in enumerate(self.learners)}
        h = self.history[self.history["learner"].isin(row)].sort_values("completed_at", kind="stable")
        self.done = np.zeros((len(self.learners), cat.n), bool)
        self.done[h["learner"].map(row).to_numpy(), h["item"].to_numpy()] = True
        self.last = np.full(len(self.learners), -1)
        self.last[h["learner"].map(row).to_numpy()] = h["item"].to_numpy()        # the last one wins
        self.order = {l: g.to_numpy() for l, g in h.groupby("learner", sort=False)["item"]}
        self.eligible = eligible(self.done, cat)
        self.segment = np.array([segment(self.order.get(l), cat) for l in self.learners])

    def __len__(self):
        return len(self.learners)


def in_progress(order, cat):
    """The courses the learner started and has not finished, the most recent first."""
    done = set(order)
    out = []
    for j in order[::-1]:
        c = cat.course[j]
        if c not in out and cat.last[c] not in done:
            out.append(c)
    return out


def segment(order, cat):
    if order is None or len(order) == 0:
        return "new"
    return "in a course" if in_progress(order, cat) else "between courses"


def top_k(scores, done, k=K):
    """The k best lessons per learner, never one the learner has done; -1 where no lesson has a score."""
    s = np.where(done, -np.inf, scores.astype(float))
    idx = np.argsort(-s, axis=1, kind="stable")[:, :k]
    return np.where(np.take_along_axis(s, idx, axis=1) > -np.inf, idx, -1)


def fill(lists, *more):
    """Fill the empty places of `lists` with the lessons of `more` (in order) that are not in the row yet."""
    out = lists.copy()
    for r in range(len(out)):
        row = [p for p in out[r] if p >= 0]
        for m in more:
            for p in m[r]:
                if len(row) == out.shape[1]:
                    break
                if p >= 0 and p not in row:
                    row.append(p)
        out[r, :len(row)] = row
    return out


def known(scores):
    """Only the lessons with a positive score can be shown (a count of 0 says nothing)."""
    return np.where(scores > 0, scores, -np.inf)


def ndcg(lists, truth):
    pos = (lists == truth[:, None])
    return (pos / np.log2(np.arange(2, lists.shape[1] + 2))).sum(axis=1)


def bootstrap(values, n=200, seed=37):
    """Mean and 95% interval: resample learners with replacement n times."""
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n, len(values)))].mean(axis=1)
    return [float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]


def evaluate(split, lists):
    """Recall@3 of the next lesson (with a 95% interval), NDCG@3, the break rate (the share of shown lessons whose
    prerequisites are not done), coverage, novelty, diversity (pairs from different courses), and recall by
    segment. Returns (results, per-learner values)."""
    cat = split.cat
    hit = (lists == split.next[:, None]).any(axis=1).astype(float)
    shown = lists >= 0
    breaks = np.where(shown, ~np.take_along_axis(split.eligible, np.maximum(lists, 0), axis=1), False)
    brk = breaks.sum(axis=1) / np.maximum(shown.sum(axis=1), 1)
    users = split.history["learner"].nunique()
    took = split.history.drop_duplicates(["learner", "item"])["item"]
    info = -np.log2((np.bincount(took, minlength=cat.n) + 1) / (users + 1))
    div = []
    for row in lists:
        row = row[row >= 0]
        if len(row) >= 2:
            c = cat.course[row]
            div.append(1 - ((c[:, None] == c[None, :]).sum() - len(row)) / (len(row) * (len(row) - 1)))
    res = {"recall": bootstrap(hit), "ndcg": float(ndcg(lists, split.next).mean()),
           "breaks": bootstrap(brk), "lists_with_break": float((brk > 0).mean()),
           "coverage": float(len(np.unique(lists[shown])) / cat.n),
           "novelty": float(info[lists[shown]].mean()), "diversity": float(np.mean(div)),
           "segments": {g: bootstrap(hit[split.segment == g]) for g in SEGMENTS},
           "segment_breaks": {g: float(brk[split.segment == g].mean()) for g in SEGMENTS}}
    return res, {"recall": hit, "breaks": brk}


def paired(a, b):
    """b - a per learner, with a 95% interval (the same learners in both)."""
    return bootstrap(np.asarray(b) - np.asarray(a))


def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Ridgeway Learning is an online learning platform (invented for this case). After each lesson, a learner's home page shows an **Up next** block with 3 lessons. Today the block shows the most popular lessons. Grace, the head of product, asks: "Can the block suggest the lesson each learner will really take next, and never a lesson they are not ready for?" Kofi, the engineer, asks: "What does it cost to run?" Grace and Kofi are made up.

Your job: compare popularity, co-occurrence of completions, a sequence method (Markov) and a prerequisite-aware rule on an honest time split. Measure next-lesson recall@3 and how often a suggestion **breaks a prerequisite**. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/what-to-learn-next/#the-situation) tells the full story.

## 2. The data

Run the next cell. It reads the catalogue and the completions, and counts them.

> **Predict.** Which course has the most completions? Why?

> **Check.** You should see 84 lessons in 12 courses, and 259,760 completions by 24,000 learners.

In [ ]:
lessons, events = read("data")
cat = Catalogue(lessons)
print(f"{cat.n} lessons in {len(cat.courses)} courses; {len(events):,} completions by {events.learner.nunique():,} learners")
print(f"from {events.completed_at.min():%d %b %Y} to {events.completed_at.max():%d %b %Y}")
per = events.groupby("learner").size()
print(f"lessons per learner: median {per.median():.0f}; only one lesson: {(per == 1).mean():.1%}")
print(events.lesson.str[:-3].value_counts().to_string())

**What this tells us.** Python basics has by far the most completions: it is the first course of most goals, and many learners stop early. 17.0% of the learners completed only one lesson.

## 3. Prerequisites: which lessons can a learner take?

A **prerequisite** is a lesson you must complete before another one. Inside a course, each lesson needs the lesson before it. The first lesson of some courses needs lessons of other courses. A lesson a learner **can take** is one they have not done, with every prerequisite done. Run the next cell.

In [ ]:
# A learner who completed the first four lessons of Python basics. Which lessons can they take?
done = np.zeros((1, cat.n), bool)
for lesson in ["PY-01", "PY-02", "PY-03", "PY-04"]:
    done[0, list(cat.ids).index(lesson)] = True
can = eligible(done, cat)[0]
print("Can take:", ", ".join(cat.ids[can]))
print("PD-01 needs:", ", ".join(cat.ids[p] for p in cat.pre[list(cat.ids).index("PD-01")]))

> **Check.** You should see `PY-05`, the first lessons of the courses with no prerequisite (`GIT-01`, `SQL-01`, `WEB-01`) and `STA-01` (Statistics needs Python lesson 4). `PD-01` needs `PY-08`, so it is not in the list.

## 4. The time split

Run the next cell. The test period is the two weeks from **19 January 2026**. The methods learn from every completion before it. For each learner who completes a lesson in the test period, we predict the **first** lesson they complete in it: their next lesson. The segments are the learner's state at the start: **new** (no lesson yet), **in a course** (a course started and not finished), **between courses** (every started course finished).

> **Check.** You should see {T['learners']:,} learners: {T['segments']['new']:,} new, {T['segments']['in a course']:,} in a course and {T['segments']['between courses']:,} between courses.

In [ ]:
test = Split(events, cat, PERIODS["test"])
print(f"{len(test):,} learners complete a lesson from {test.start:%d %b %Y}; history: {len(test.history):,} completions")
print(pd.Series(test.segment).value_counts().reindex(SEGMENTS).to_string())

> **Your turn.** Real learners do not always follow the prerequisites. What share of the learners' real next lessons (`test.next`, one lesson number per learner) were lessons they **could take** (`test.eligible`, one row of True or False per learner)? Store it in `share_ready`.

In [ ]:
# Your answer: the share of real next lessons that the learner could take.
share_ready = ...

In [ ]:
expect('share_ready', rounded(share_ready), 0.945, tolerance=0.001)

**What this tells us.** 94.5% of the next lessons were lessons the learner could take. The other 5.5% broke a prerequisite: in our behaviour model, some learners jump ahead. A method that never breaks a prerequisite can never find those.

## 5. The baseline: popular lessons

Run the next cell. `popular` ranks lessons by the number of learners who completed them in the 28 days before the period. Each learner gets the most popular lessons they have not done.

> **Predict.** What will the popular list show to a new learner?

In [ ]:
def popular(split, days=28):
    """Lessons ranked by the number of learners who completed them in the last `days` before the period."""
    h = split.history
    recent = h[h["completed_at"] >= split.start - pd.Timedelta(days=days)]
    return np.bincount(recent["item"], minlength=split.cat.n).astype(float)


results, per_learner = {}, {}
pop = popular(test)
everything = top_k(np.tile(pop, (len(test), 1)), test.done, k=cat.n)     # every lesson not done, most popular first
lists = {"popular": fill(top_k(np.tile(pop, (len(test), 1)), test.done))}
results["popular"], per_learner["popular"] = evaluate(test, lists["popular"])
new_learner = np.flatnonzero(test.segment == "new")[0]
print("A new learner sees:", ", ".join(cat.ids[lists["popular"][new_learner]]))
print(f"popular  recall@3 {results['popular']['recall'][0]:.3f}  break rate {results['popular']['breaks'][0]:.1%}")

> **Check.** You should see `PY-01, PY-02, PY-03`, recall@3 0.415 and a break rate of 58.5%.

A new learner can take `PY-01`, but not `PY-02` or `PY-03`: two of the three cards break a prerequisite.

## 6. Co-occurrence

"Learners who completed this also completed that." Run the next cell. It counts, for each pair of lessons, the learners who completed both (item-item cosine, as in the item-item lesson). A learner's score for a lesson is the sum of its similarities with the lessons they have done.

In [ ]:
def cooccurrence(split):
    """Item-item cosine between lessons: learners who completed both, divided by the square root of the product of
    the two counts (as in item-item collaborative filtering). The diagonal is 0."""
    h = split.history.drop_duplicates(["learner", "item"])
    users = {l: r for r, l in enumerate(h["learner"].unique())}
    m = np.zeros((len(users), split.cat.n))
    m[h["learner"].map(users).to_numpy(), h["item"].to_numpy()] = 1
    both = m.T @ m
    n = np.diag(both).copy()
    sim = both / np.sqrt(np.outer(n, n) + 1e-9)
    np.fill_diagonal(sim, 0)
    return sim


C = cooccurrence(test)
lists["co-occurrence"] = fill(top_k(known(test.done.astype(float) @ C), test.done), everything)
results["co-occurrence"], per_learner["co-occurrence"] = evaluate(test, lists["co-occurrence"])
print(f"co-occurrence  recall@3 {results['co-occurrence']['recall'][0]:.3f}  break rate {results['co-occurrence']['breaks'][0]:.1%}")

> **Check.** You should see recall@3 0.726 and a break rate of 52.3%.

## 7. Markov: what learners took next

Run the next cell. It counts, for each lesson, which lesson the same learner completed **directly after** it (a Markov model, as in [Sessions, sequences and LLMs, briefly](https://learning.nextia-ai.com/courses/recommendations/m07/sessions-sequences-and-llms-briefly/)). A learner gets the lessons that most often followed their **last** lesson.

In [ ]:
def transitions(split):
    """Markov: how often lesson b was the next lesson completed after lesson a, by the same learner."""
    h = split.history.sort_values(["learner", "completed_at"], kind="stable")
    a, b = h["item"].to_numpy()[:-1], h["item"].to_numpy()[1:]
    same = h["learner"].to_numpy()[:-1] == h["learner"].to_numpy()[1:]
    T = np.zeros((split.cat.n, split.cat.n))
    np.add.at(T, (a[same], b[same]), 1)
    return T


def markov_scores(split, T):
    """Score each lesson by how often it followed the learner's last lesson. New learners get zeros."""
    s = np.zeros((len(split), split.cat.n))
    has = split.last >= 0
    s[has] = T[split.last[has]]
    return s


T = transitions(test)
lists["Markov"] = fill(top_k(known(markov_scores(test, T)), test.done), everything)
results["Markov"], per_learner["Markov"] = evaluate(test, lists["Markov"])
print(f"Markov  recall@3 {results['Markov']['recall'][0]:.3f}  break rate {results['Markov']['breaks'][0]:.1%}")

> **Check.** You should see recall@3 0.853 and a break rate of 44.4%.

Markov knows the order, so it finds the next lesson much more often. But after the first place, its counts are small, and many of its suggestions are lessons the learner is not ready for.

## 8. The prerequisite rule

Run the next cell. The rule shows only lessons the learner can take. First the next lessons of the courses in progress (the most recent course first). Then the first lessons of new courses, most popular first. It takes a few seconds.

In [ ]:
def rank_by(pop, after=None):
    """A number between 0 and 1 for each lesson, to order the new courses: by popularity, or (with `after`, the
    Markov counts of the learner's last lesson) by Markov first and popularity to break ties."""
    p = pop / (pop.max() + 1)
    if after is None:
        return p
    return (after + p) / (after.max() + 2)


def rule_scores(order, cat, rank):
    """The prerequisite-aware rule for one learner (`order`: the lessons done, oldest first). Only lessons whose
    prerequisites are done get a score: first the next lessons of the courses in progress (the most recent course
    first, then by position), then the first lessons of new courses (by `rank`), then the rest (optional lessons
    left behind), by `rank`. Other lessons get -inf."""
    done = np.zeros((1, cat.n), bool)
    done[0, list(order)] = True
    ok = eligible(done, cat)[0]
    s = np.where(ok, rank, -np.inf)
    progress = in_progress(list(order), cat)
    for j in np.flatnonzero(ok):
        c = cat.course[j]
        if c in progress:
            s[j] = 100 - 10 * progress.index(c) - cat.position[j] / 100
        elif j == cat.first[c]:
            s[j] = 2 + rank[j]
    return s


def rule(split, pop, T=None):
    """The rule for every learner; with T (Markov counts), new courses are ordered by what followed the last lesson."""
    out = []
    for l, last in zip(split.learners, split.last):
        after = T[last] if T is not None and last >= 0 else None
        out.append(rule_scores(split.order.get(l, []), split.cat, rank_by(pop, after)))
    return np.array(out)


lists["prerequisite rule"] = fill(top_k(rule(test, pop), test.done), everything)
results["prerequisite rule"], per_learner["prerequisite rule"] = evaluate(test, lists["prerequisite rule"])
print(f"rule  recall@3 {results['prerequisite rule']['recall'][0]:.3f}  break rate {results['prerequisite rule']['breaks'][0]:.1%}")

> **Check.** You should see recall@3 0.910 and a break rate of 0.2%.

## 9. Rule + Markov, and the comparison

The rule knows what a learner **can** take. Markov knows what learners **did** take after a lesson. For learners between courses, the rule orders the new courses by popularity; Markov can order them by what followed the learner's last lesson. `rule(test, pop, T)` does that.

> **Your turn.** Make the lists **rule + Markov**: `rule(test, pop, T)`, then `top_k(..., test.done)`, then `fill(..., everything)`. Evaluate them. Store the lists in `design` and recall@3 in `design_recall` (`evaluate` returns the results first; recall is `["recall"][0]`).

In [ ]:
# Your answer: the rule, with new courses ordered by Markov.
design = ...
design_recall = ...

In [ ]:
expect('design_recall', rounded(design_recall), 0.919, tolerance=0.001)

Run the next cell. It makes the rule + Markov lists again (so the table works even if your answer above is not finished) and shows every method in one table, on the same learners with the same measures.

In [ ]:
def table(results):
    """One row per method: recall@3 (95% interval), NDCG@3, break rate, coverage, novelty, diversity."""
    rows = []
    for name, o in results.items():
        rows.append({"method": name, "recall@3": round(o["recall"][0], 3),
                     "interval": f"{o['recall'][1]:.3f} to {o['recall'][2]:.3f}", "NDCG@3": round(o["ndcg"], 3),
                     "break rate": f"{o['breaks'][0]:.1%}", "coverage": f"{o['coverage']:.0%}",
                     "novelty": round(o["novelty"], 2), "diversity": round(o["diversity"], 2)})
    return pd.DataFrame(rows).set_index("method")


lists["rule + Markov"] = fill(top_k(rule(test, pop, T), test.done), everything)
results["rule + Markov"], per_learner["rule + Markov"] = evaluate(test, lists["rule + Markov"])
table(results)

> **Check.** You should see rule + Markov first (0.919), then the rule (0.910), Markov (0.853), co-occurrence (0.726) and popular (0.415). Only the two rules have a break rate near 0.

Run the next cell. It shows recall@3 by segment.

In [ ]:
pd.DataFrame({name: {g: round(results[name]["segments"][g][0], 3) for g in SEGMENTS} for name in results})

**What this tells us.** In a course, the rule is almost as good as the design: the next lesson is almost always the next one in the course. Between courses, the rule alone falls to 0.794, and Markov's order raises it to 0.926. For new learners, every method that knows prerequisites shows the first lessons of the courses with no prerequisite.

## 10. Failure case: co-occurrence ignores order

Run the next cell. It shows the lists of one learner who has just completed `PD-03`. Each lesson is marked `ready` or `NOT READY`.

In [ ]:
row = list(test.learners).index('R01110')
print("Done:", ", ".join(cat.ids[test.order[test.learners[row]]]))
print("Really next:", cat.ids[test.next[row]])
for name in ["popular", "co-occurrence", "Markov", "rule + Markov"]:
    print(f"{name:14}", ", ".join(f"{cat.ids[p]} ({'ready' if test.eligible[row, p] else 'NOT READY'})" for p in lists[name][row]))

> **Check.** For co-occurrence you should see `PD-04 (ready), PD-05 (NOT READY), SQL-02 (NOT READY)`.

**What this tells us.** Learners who completed `PD-03` also completed `PD-05`: but **after** `PD-04`. Co-occurrence counts the pair and forgets the order. It also suggests `SQL-02` because data analysts take both courses, although this learner has not started SQL.

## 11. End to end on new input: 2 to 15 February

The last two weeks were never used. Run the next cell. It does the whole design again: the Markov counts and popularity from every completion before 2 February, then the lists for the learners of the new period.

> **Check.** You should see 5,000 learners and recall@3 0.917 for rule + Markov (popular 0.412).

In [ ]:
new = Split(events, cat, PERIODS["new input"])
pop_new, T_new = popular(new), transitions(new)
everything_new = top_k(np.tile(pop_new, (len(new), 1)), new.done, k=cat.n)
final = fill(top_k(rule(new, pop_new, T_new), new.done), everything_new)
check = {"popular": fill(top_k(np.tile(pop_new, (len(new), 1)), new.done)), "rule + Markov": final}
print(f"{len(new):,} learners complete a lesson from {new.start:%d %b %Y}")
table({name: evaluate(new, l)[0] for name, l in check.items()})

A service does not make lists for everyone at once. It gets one learner when they open their home page. Run the next cell. `suggest_one` makes the list for one learner from the lessons they completed, in order.

In [ ]:
def suggest_one(order, T, pop, cat, k=K):
    """The design for one learner, as a service makes it: `order` is the lessons done, oldest first. The rule, with
    new courses ordered by Markov from the last lesson; then popular lessons if fewer than k can be taken."""
    done = np.zeros((1, cat.n), bool)
    done[0, list(order)] = True
    after = T[order[-1]] if len(order) else None
    first = top_k(rule_scores(order, cat, rank_by(pop, after))[None, :], done, k)
    return fill(first, top_k(pop[None, :], done, k))[0].tolist()


row = list(new.learners).index('R01025')
order = list(new.order[new.learners[row]])
print("Done:", ", ".join(cat.ids[order]))
start = time.perf_counter()
up_next = suggest_one(order, T_new, pop_new, cat)
print(f"Up next: {', '.join(cat.ids[up_next])}  (made in {(time.perf_counter() - start) * 1000:.1f} ms)")
print("Really next:", cat.ids[new.next[row]])
print("Same as the batch list:", up_next == final[row].tolist())

> **Check.** You should see `Up next: STA-01, GIT-01, PD-01`, really next `STA-01`, and `True`.

## 12. Recap

You split synthetic progress data by time and compared popularity, co-occurrence, a Markov sequence method and a prerequisite rule on the same learners. You measured next-lesson recall@3 and the break rate, by segment. Order and prerequisites mattered much more than popularity: the rule kept every suggestion ready, and Markov ordered the new courses. The data is synthetic, so the methods found what our behaviour model put in. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/what-to-learn-next/#the-recommendation) turns this into a recommendation with its costs, its limits and what a person still checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/recommendations/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m10/what-to-learn-next/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is synthetic: made for this case study by a documented behaviour model. No real learners.